In [1]:
import re
import os
import streamlit as st
import yfinance as yf
import plotly.graph_objects as go
import plotly.express as px
from phi.agent import Agent
from phi.model.groq import Groq
from phi.tools.yfinance import YFinanceTools
from phi.tools.duckduckgo import DDGS
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
def extract_data(ticker, period="6mo"):
    stock = yf.Ticker(ticker)
    hist = stock.history(period=period)    
    hist.reset_index(inplace=True)
    return hist


def plot_stock_price(hist, ticker):
    fig = px.line(hist, x="Date", y="Close", title=f"{ticker} Stock Prices (Last 6 Months)", markers=True)    
    st.plotly_chart(fig)

def plot_candlestick(hist, ticker):
    fig = go.Figure(data=[go.Candlestick(x=hist['Date'], open=hist['Open'], high=hist['High'], low=hist['Low'], close=hist['Close'])])
    fig.update_layout(title=f"{ticker} Candlestick Chart (Last 6 Months)")
    st.plotly_chart(fig)

def plot_moving_average(hist, ticker):
    # Calculates the 20-period Simple Moving Average (SMA) and adds it to the DataFrame.
    hist['SMA_20'] = hist['Close'].rolling(window=20).mean()
    # Calculates the 20-period Exponential Moving Average (EMA) and adds it to the DataFrame.
    hist['EMA_20'] = hist['Close'].ewm(span=20, adjust=False).mean()
    
    fig = px.line(hist, x='Date', y=['Close', 'SMA_20', 'EMA_20'], title=f"{ticker} Moving Average (Last 6 Months)", labels={'value': 'Price (USD)', 'Date': 'Date'})
    st.plotly_chart(fig)

def plot_volume(hist, ticker):
    fig = px.bar(hist, x='Date', y='Volume', title=f"{ticker} Trading Volume (Last 6 Months)")    
    st.plotly_chart(fig)

def search_web(query: str) -> str:
    """Searches DuckDuckGo for the given query and returns results."""
    with DDGS() as ddgs:
        results = [r for r in ddgs.text(query, max_results=5)]
        return str(results)

In [4]:
web_search_agent = Agent(name="Web Search Agent",
                              role="To search the web",
                              model=Groq(id="openai/gpt-oss-120b"),
                              tools=[search_web],
                              instructions=["Always includes the sources"],
                              show_tool_calls=True, markdown=True)

financial_agent = Agent(name="Financial Agent",
                              model=Groq(id="openai/gpt-oss-120b"),
                              tools=[YFinanceTools(stock_price=True,
                                                   analyst_recommendations=True,
                                                   stock_fundamentals=True,
                                                   company_news=True)],
                              instructions=["Use tables to show the data"],
                              show_tool_calls=True, markdown=True)

multi_ai_agent = Agent(team=[web_search_agent, financial_agent],
                       model=Groq(id="llama-3.3-70b-versatile"),
                       instructions=["Always include sources", "Use tables to show the data"],
                       show_tool_calls=True, markdown=True)


In [5]:
ticker = "GOOG"

In [6]:
hist = extract_data(ticker)

In [7]:
ai_response = multi_ai_agent.run(f"Summarize the analyst recomendation and share the last news about {ticker}")

In [8]:
print(ai_response)

content='\nRunning:\n - transfer_task_to_financial_agent(additional_information=GOOG ticker symbol, expected_output=A summary of analyst recommendations for GOOG, task_description=Get analyst recommendations for GOOG)\n - transfer_task_to_financial_agent(additional_information=GOOG ticker symbol, expected_output=The last news about GOOG, task_description=Get company news for GOOG)\n\n## Summary of Analyst Recommendation and Latest News for GOOG\n\n### Summary of Analyst Recommendation:\n\nThe analyst community is overwhelmingly positive on Alphabet (GOOG), with the majority recommending “Buy” or “Strong Buy” and **zero** sell signals in the recent three-month window. This suggests strong confidence in the stock’s near- to medium-term performance. The proportion of “Buy” recommendations has nudged upward, while “Strong Buy” has modestly declined.\n\n### Latest News:\n\nAlphabet announced plans to raise up to $80 billion in equity to support its artificial-intelligence initiatives, inclu